# Test models with CIFAR10 

In [118]:
import math
import os
import random
import time
from datetime import datetime

import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd

from typing import Sequence


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [119]:
# Seed every RNG so runs are comparable: CIFAR-10 seed variance (~±0.3-0.7%)
# is the same magnitude as many single-change effects.
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

## Model

### ViT

In [120]:
class PatchEmbedding(nn.Module):
    """
        Convert image -> patches -> Conv2d to learn the features on each patch.
    
        Given B= batch_size, C= in_channels, H= image_height, W= image_width, ps= patch_size, D= embed_dim.
    
        => (B, C, H, W) -> (B, D, H//ps, W//ps) -> (B, D, num_patches) -> (B, num_patches, D).
    
        Returns (B, num_patches, D).
            
    """
    def __init__(self, 
                 img_size, # Size of the input image (H, W)
                 patch_size, # Size of each patch
                 in_channels, # Number of input channels
                 embed_dim): # Dimension of the embedding (number of learning features learnt from each patch)
        
        super().__init__()

        assert img_size % patch_size == 0

        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2 # num_patches = H//ps * W//ps

        # Patch tokens
        # Create vectors learning features from patches
        self.projection = nn.Sequential(

            # (B, C, H, W) -> (B, D, H//ps, W//ps)
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=embed_dim,
                kernel_size=patch_size,
                stride=patch_size, # stride = kernel_size -> No overlapping
            ),

            # Convert patch grid -> sequence
            # (B, D, H//ps, W//ps) -> (B, D, num_patches)
            nn.Flatten(start_dim=2),
        )

        # Classification tokens
        # Update and learn throughout the training
        # -> (1, 1, D)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Position tokens
        # Positional info for each token
        # -> (1, 1 + num_patches, D)
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + self.num_patches, embed_dim))

        # Init CLS and positions with smaller std (randn std too close to 1 to start with)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        # Batch size (B, C, H, W = x.shape)
        B = x.size(0)

        # Pass foward the patch tokenization
        x = self.projection(x)

        # (B, D, num_patches) -> (B, num_patches, D)
        # Swap 1 and 2 dim toegther
        x = x.transpose(1, 2)
        # Other way, reorder everything: x = x.permute(0, 2, 1)

        # Create tokens with same size as x based on self.cls_token
        # -> (B, 1, D)
        cls_tokens = self.cls_token.expand(B, -1, -1)

        # Add CLS token -> (B, 1 + num_patches, D)
        x = torch.cat((cls_tokens, x), dim=1)

        # Add positions -> (B, 1 + num_patches, D)
        x += self.pos_embed

        return x
        

In [ ]:
def scaled_dot_product(q, k, v, # Query, Key, Value
                        mask=None, # if masking, irrelevant position got set as neg_inf -> after softmax norm: val = 0
                        dropout=None):

    # Number of values of key/query vectors
    # Use to normalise the attention scores -> more stable, more focus on learning without being overwhemled
    d_k = q.size()[-1] 

    # scores = (Q @ K.T)/math.sqrt(d_k)
    # (B, H, N, D) @ (B, H, D, N)
    # -> (B, H, N, N)
    # matmul = more than 2D, dot = classic 1D or linear alg vec projections
    scores = torch.matmul(
        q,
        k.transpose(-2, -1),
    ) / math.sqrt(d_k)

    if mask is not None: scores += mask

    # Attention prob across keys (range from 0 to 1)
    # attention = softmax(scores)
    attention = F.softmax(scores, dim=-1)

    if dropout is not None:
        attention = dropout(attention)

    # values = attention * V
    # (B, H, N, N) @ (B, H, N, D)
    # -> (B, H, N, D)
    values = torch.matmul(attention, v)

    return values, attention

class MultiHeadSelfAttention(nn.Module):
    """
        MSA: Split data into smaller parts and run multiple attention tests simultaneously.
    """
    def __init__(self, 
                 embed_dim, # Dimension of the embedding
                 num_heads, # Number of heads
                 dropout=0.0): # Dropout rate
        
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.head_dim = embed_dim // num_heads # Dimension per head

        self.qkv_layer = nn.Linear(embed_dim, 3 * embed_dim)

        self.out_projection_layer = nn.Linear(embed_dim, embed_dim)

        self.attention_dropout = nn.Dropout(dropout)
        self.output_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None, return_attention=False):
        B, N, E = x.shape

        # x -> concat(q, k, v)
        # (B, N, E) → (B, N, 3E)
        qkv = self.qkv_layer(x)

        # Reshape to (B, N, 3, H, D)
        # (B, N, 3E) → (B, N, 3, H, D)
        qkv = qkv.reshape(
            B,
            N,
            3,
            self.num_heads,
            self.head_dim,
        )

        # Rearrange (B, N, 3, H, D) -> (3, B, H, N, D)
        # Rearrange the whole thing
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv.unbind(dim=0)

        # Apply scaled_dot_product
        values, attention = scaled_dot_product(q, k, v, mask, dropout=self.attention_dropout)

        # Combine attention heads

        # (B, H, N, D) -> (B, N, H, D)
        output = values.transpose(1, 2)

        # Reshape (B, N, H, D) -> (B, N, E)
        output = output.reshape(B, N, self.num_heads * self.head_dim)

        output = self.out_projection_layer(output)
        output = self.output_dropout(output)

        if return_attention: return output, attention

        return output

In [122]:
class MLP(nn.Module):
    """
        MLP = Multilayer Perceptron.

        Processes and transforms the features within each token.
    """
    def __init__(self, 
                 in_features, # Number of features input
                 hidden_features, # Number of hidden features
                 drop_rate): # Dropout rate

        super().__init__()
        
        self.feat_layers = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.GELU(),
            nn.Dropout(drop_rate),
            nn.Linear(hidden_features, in_features),
            nn.Dropout(drop_rate)

        )

    def forward(self, x):
        x = self.feat_layers(x)

        return x
        

In [123]:
# Pre-LayerNorm applies LayerNorm before both the attention and feed-forward blocks. 
# This stabilizes gradient flow and prevents the exploding/vanishing gradient problem in deep Transformers.

class TransformerEncoderLayer(nn.Module):
    """
        Transformer Encoder block.

        ((Norm -> MSA) + X -> Norm -> MLP) + X -> out
    """
    def __init__(self, embed_dim, num_heads, mlp_dim, drop_rate):
        super().__init__()

        self.ln1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=drop_rate, batch_first=True)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.MLP = MLP(embed_dim, mlp_dim, drop_rate=drop_rate)

    def forward(self, x):
        h = self.ln1(x)
        attn_out, _ = self.attention(h, h, h, need_weights=False)
        x = x + attn_out
        x = x + self.MLP(self.ln2(x))

        return x
    

In [124]:
class ViT(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, num_classes, embed_dim, depth, num_heads, mlp_dim, drop_rate):
        super().__init__()

        self.ViT_layers = nn.Sequential(
            PatchEmbedding(img_size, patch_size, in_channels, embed_dim),

            *[
                TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, drop_rate)
                for _ in range(depth)
            ],

            nn.LayerNorm(embed_dim),

        )

        self.head = nn.Linear(embed_dim, num_classes)

        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self):
        n_params = sum(p.numel() for p in self.parameters())
        return n_params
    
    def forward(self, x):
        x = self.ViT_layers(x)
        cls_tokens = x[:, 0]

        return self.head(cls_tokens)

### Resnet

In [125]:
# class BasicResidualBlock(nn.Module):
#     """
#         Canonical basic block (for resnet18, resnet34), He et al. 2015:

#         input > 3×3 Conv (stride) > BN > ReLU > 3×3 Conv (1) > BN > Add(identity) > ReLU

#         Convs carry no bias: the BatchNorm right after each conv has its own shift,
#         so a conv bias would be a dead parameter.

#         *In this case, the cifar10 dataset is not complicated enough for the bottleneck block to make any differences. > use basic block for now.*
#     """

#     expansion = 1

#     def __init__(self, 
#                  input_dim: int, # number of channels
#                  planes: int, # internal width per stage in net
#                  stride: int = 1, # initial stride    
#                  downsample: nn.Module = None, # block of shortcut
#         ):
#         super(BasicResidualBlock, self).__init__()

#         output_channels = planes * self.expansion

#         self.residual = nn.Sequential(
#             nn.Conv2d(input_dim, planes, kernel_size=3, stride=stride, padding=1, bias=False),
#             nn.BatchNorm2d(planes),
#             nn.ReLU(inplace=True),

#             nn.Conv2d(planes, output_channels, kernel_size=3, stride=1, padding=1, bias=False),
#             nn.BatchNorm2d(output_channels),
#         )

#         self.downsample = downsample if downsample else nn.Identity()
    
#     def forward(self, x):
#         identity = self.downsample(x)
#         residual = self.residual(x)
#         # Post-add activation: out = relu(F(x) + x). The nonlinearity must sit
#         # AFTER the addition, otherwise blocks can only ever add non-negative values.
#         return F.relu(identity + residual)

# class resnet(nn.Module):
#     def __init__(self,
#         Block,
#         layers: Sequence[int],
#         num_classes: int = 10,
#         input_channels: int = 3):

#         super(resnet, self).__init__()

#         self.in_channels = 64
#         # CIFAR stem (He et al. sec 4.2): 3×3 stride-1 conv, no maxpool.
#         # The ImageNet stem (7×7 s2 + maxpool) would shrink 32×32 inputs to 8×8
#         # before the first residual block, leaving the deep stages with 2×2/1×1 maps.
#         self.conv1 = nn.Sequential(
#             nn.Conv2d(input_channels, 64, kernel_size=3, stride=1, padding=1, bias=False),
#             nn.BatchNorm2d(64),
#             nn.ReLU(inplace=True),
#         )

#         self.maxpool = nn.Identity()

#         # Layers — stage resolutions on CIFAR: 32 > 16 > 8 > 4
#         self.big_layers = nn.Sequential(
#             self._make_layer(Block=Block, planes=64, number_of_blocks=layers[0], stride=1),
#             self._make_layer(Block=Block, planes=128, number_of_blocks=layers[1], stride=2),
#             self._make_layer(Block=Block, planes=256, number_of_blocks=layers[2], stride=2),
#             self._make_layer(Block=Block, planes=512, number_of_blocks=layers[3], stride=2),   
#         )

#         self.avgpool = nn.AdaptiveAvgPool2d((1,1))

#         # Single linear head: the 512-d pooled vector is already a summary,
#         # a deep MLP here is pure memorization capacity.
#         self.classification_head = nn.Linear(512 * Block.expansion, num_classes)

#         # Kaiming fan-out init for ReLU conv stacks (PyTorch default is fan-in
#         # with a=sqrt(5), which under-scales). BN starts at gamma=1, beta=0.
#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
#             elif isinstance(m, nn.BatchNorm2d):
#                 nn.init.ones_(m.weight)
#                 nn.init.zeros_(m.bias)

#         # Zero-init the LAST BN gamma of each block so every block starts as an
#         # identity mapping (Goyal et al. 2017) — stabilises early high-LR training.
#         for m in self.modules():
#             if isinstance(m, BasicResidualBlock):
#                 nn.init.zeros_(m.residual[4].weight)

#     def _make_layer(self,
#         Block: BasicResidualBlock,
#         planes: int,
#         number_of_blocks: int,
#         stride: int = 1):

#         output_channels = planes * Block.expansion

#         layers = []
#         downsample = None

#         if stride != 1 or self.in_channels != output_channels:
#             # Shortcut projection uses the SAME normalization as the residual
#             # branch (BN), so both paths stay in one statistics regime.
#             downsample = nn.Sequential(
#                 nn.Conv2d(self.in_channels, output_channels, kernel_size=1, stride=stride, padding=0, bias=False),
#                 nn.BatchNorm2d(output_channels),
#             )

#         layers.append(Block(input_dim=self.in_channels, 
#                          planes=planes, 
#                          downsample=downsample,
#                          stride=stride))

#         self.in_channels = output_channels

#         for i in range(1, number_of_blocks):
#             layers.append(Block(self.in_channels, planes=planes))

#         return nn.Sequential(*layers)
    
#     def forward(self, x):
#         x = self.conv1(x)
#         x = self.maxpool(x)
#         x = self.big_layers(x)
#         x = self.avgpool(x)
#         x = torch.flatten(x, 1)  # Flatten the tensor
#         return self.classification_head(x)


In [126]:
# model = resnet(BasicResidualBlock, [3, 4, 6, 3])
# x = torch.randn(4, 3, 32, 32)
# output = model(x)

# print(output.shape)
# # torch.Size([4, 10])

### Basic CNN

In [127]:
# # input -> conv -> relu -> pool -> conv -> relu -> pool -> fc -> softmax (output)
# class CNN(nn.Module):
#     """
#     Parameters:
#            * in_channels: Number of channels in the input image (for grayscale images, 1)
#            * num_classes: Number of classes to predict.
           
#     """
#     def __init__(self, input_dim: int, output_dim: int):
#         super(CNN, self).__init__()

#         self.model = nn.Sequential(
#             nn.Conv2d(input_dim, 16, kernel_size=3, stride=1, padding=1), # Conv 1
#             # nn.BatchNorm2d(16),
#             nn.GELU(),

#             nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

#             nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # Conv 2
#             # nn.BatchNorm2d(32),
#             nn.GELU(),
#             nn.Dropout(0.2),

#             nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1), # Conv 3
#             # nn.BatchNorm2d(64),
#             nn.GELU(),
#             nn.Dropout(0.3),

#             nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

#             nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1), # Conv 4 
#             # nn.BatchNorm2d(64),
#             nn.GELU(),

#         )
#         self.fc1 = nn.Linear(64 * 4 * 4, 128)
#         self.fc2 = nn.Linear(128, output_dim) # Output layer, output_dim = number of classes output

#     def forward(self, x):
#         """
#             Model flow: input -> conv -> relu -> pool -> conv -> relu -> pool -> fc -> softmax (output)
#         """
#         x = self.model(x)
#         x = x.reshape(x.shape[0], -1)  # Flatten the tensor
#         # print(x.shape)
#         x = F.gelu(self.fc1(x)) # Apply fully connected layer 
#         x = self.fc2(x) # Output layer
#         return x

## Data Loading

In [128]:
# Hyperparameters
BATCH_SIZE = 128
EPOCHS = 200
LEARNING_RATE = 3e-4
PATCH_SIZE = 4
NUM_CLASSES = 10
IMAGE_SIZE = 32
CHANNELS = 3
EMBED_DIM = 256
NUM_HEADS = 8
DEPTH = 6
MLP_DIM = 512
DROP_RATE = 0.1
WEIGHT_DECAY = 5e-2
WARMUP_EPOCHS = 10

In [129]:
# Per-channel statistics of the CIFAR-10 train split (published values).
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    # Geometric augmentation happens in PIL space, BEFORE ToTensor.
    transforms.RandomCrop(32, padding=4, padding_mode='reflect'),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    # Cutout-style occlusion on the normalized tensor — attacks memorization.
    # transforms.RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=0),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

train_dataset = datasets.CIFAR10(root='dataset/', train=True, transform=train_transform, download=True)
test_dataset = datasets.CIFAR10(root='dataset/', train=False, transform=test_transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True,
                          generator=torch.Generator().manual_seed(SEED))
test_loader = DataLoader(dataset=test_dataset, batch_size=256, shuffle=False,
                         num_workers=4, pin_memory=True, persistent_workers=True)

# Fixed CLEAN 10k subset of the train split for measuring train accuracy:
# evaluating on the augmented train_loader understates it (and a full 50k
# pass every epoch is 5x the eval cost for no extra signal).
eval_train_dataset = datasets.CIFAR10(root='dataset/', train=True, transform=test_transform, download=True)
eval_train_loader = DataLoader(torch.utils.data.Subset(eval_train_dataset, range(10000)),
                               batch_size=512, shuffle=False, num_workers=4, pin_memory=True)

In [130]:
train_dataset[0][0].shape

torch.Size([3, 32, 32])

In [131]:
len(set(train_dataset.targets))

10

## Training

In [132]:
input_dim = train_dataset[0][0].shape[0]
img_size = train_dataset[0][0].shape[1]
output_dim = len(set(train_dataset.targets))

In [133]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

torch.backends.cudnn.benchmark = True  # fixed input size -> let cudnn pick the fastest kernels

set_seed()  # seed right before weight init so the run is reproducible

cuda NVIDIA GeForce RTX 5080


### ViT

In [134]:
vit = ViT(IMAGE_SIZE, PATCH_SIZE, CHANNELS, NUM_CLASSES,
    EMBED_DIM, DEPTH, NUM_HEADS, MLP_DIM, DROP_RATE
).to(device)

vit = vit.to(memory_format=torch.channels_last)

number of parameters: 3.20M


In [135]:
vit

ViT(
  (ViT_layers): Sequential(
    (0): PatchEmbedding(
      (projection): Sequential(
        (0): Conv2d(3, 256, kernel_size=(4, 4), stride=(4, 4))
        (1): Flatten(start_dim=2, end_dim=-1)
      )
    )
    (1): TransformerEncoderLayer(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (attention): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (MLP): MLP(
        (feat_layers): Sequential(
          (0): Linear(in_features=256, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.1, inplace=False)
          (3): Linear(in_features=512, out_features=256, bias=True)
          (4): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (2): TransformerEncoderLayer(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=Tru

In [136]:
# Label smoothing caps logit over-confidence
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

decay, no_decay = [], []
for name, p in vit.named_parameters():
    (no_decay if p.ndim == 1 else decay).append(p)

optimizer = optim.AdamW(vit.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# 5-epoch linear warmup (0.01 -> 0.1) guards against early divergence at lr=0.1,
# then cosine decay to 0. Stepped once per epoch in run().
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - 5, eta_min=0.0)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

### Resnet

In [137]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

# torch.backends.cudnn.benchmark = True  # fixed input size -> let cudnn pick the fastest kernels

# set_seed()  # seed right before weight init so the run is reproducible
# net = resnet(Block=BasicResidualBlock, 
#              layers=[3, 4, 6, 3],  
#              num_classes=output_dim,
#              input_channels=input_dim).to(device)
# net = net.to(memory_format=torch.channels_last)
# # net = torch.compile(net, fullgraph=True)

# print(f"{sum(p.numel() for p in net.parameters()) / 1e6:.2f}M parameters")

In [138]:
# net

In [139]:
# epochs = 100

# # Label smoothing caps logit over-confidence; the printed loss will floor
# # near ~0.5 instead of 0 — that is expected, not a bug.
# criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# # SGD + momentum is the canonical CIFAR ResNet recipe (the previous AdamW runs
# # showed the classic Adam generalization deficit: train 92.5% / test 82.5%).
# # Weight decay goes on conv/linear weights ONLY: decaying BN affine params and
# # biases (everything with ndim == 1) fights the normalization instead of regularizing.
# decay, no_decay = [], []
# for name, p in net.named_parameters():
#     (no_decay if p.ndim == 1 else decay).append(p)

# # optimizer = torch.optim.SGD(
# #     [{"params": decay, "weight_decay": 5e-4},
# #      {"params": no_decay, "weight_decay": 0.0}],
# #     lr=0.1,  # calibrated to BATCH_SIZE = 128
# #     momentum=0.9,
# #     nesterov=True,
# # )

# optimizer = torch.optim.AdamW(net.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# # 5-epoch linear warmup (0.01 -> 0.1) guards against early divergence at lr=0.1,
# # then cosine decay to 0. Stepped once per epoch in run().
# warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=5)
# cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - 5, eta_min=0.0)
# scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[5])

### Basic CNN

In [140]:
# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter("runs/cnn_cifar10")

In [141]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device)
# model = CNN(input_dim=input_dim, output_dim=output_dim).to(device)

In [142]:
# epochs = 30
# criterion = nn.CrossEntropyLoss() # Apply softmax as an activation func for the output layer
# optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

In [143]:
# for param_tensor in model.state_dict():
#     print(param_tensor, "\t", model.state_dict()[param_tensor].size())

In [144]:
# model.eval

### Accuracy

In [145]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
            y = y.to(device, non_blocking=True)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

    return num_correct / num_samples

### Run

In [146]:
def run(model, optimizer, scheduler, criterion, epochs, writer, run_name):
    # AMP: forward in reduced precision, gradients rescaled to avoid underflow.
    # enabled= flags keep the whole loop runnable on CPU too.
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    os.makedirs("checkpoints", exist_ok=True)
    best_acc = 0.0
    step = 0

    try:
        for epoch in range(epochs):
            t0 = time.perf_counter()
            running_loss = 0.0
            model.train()

            for data, targets in train_loader:
                data = data.to(device, non_blocking=True, memory_format=torch.channels_last)
                targets = targets.to(device, non_blocking=True)

                # Forward propagation
                with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                    scores = model(data)
                    loss = criterion(scores, targets)

                # Backward propagation
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item()

                if step % 50 == 0:
                    writer.add_scalar("Loss/train_batch", loss.item(), step)
                step += 1

            avg_epoch_loss = running_loss / len(train_loader)

            # Train accuracy on the clean fixed subset, test on the full test set.
            train_acc = check_accuracy(eval_train_loader, model)
            test_acc = check_accuracy(test_loader, model)

            # Step the scheduler once per epoch
            scheduler.step()

            # Log the learning rate used for the next epoch
            current_lr = optimizer.param_groups[0]["lr"]

            epoch_time = time.perf_counter() - t0
            writer.add_scalar("Loss/train_epoch", avg_epoch_loss, epoch)
            writer.add_scalar("Accuracy/train", train_acc, epoch)
            writer.add_scalar("Accuracy/test", test_acc, epoch)
            writer.add_scalar("Accuracy/gap", train_acc - test_acc, epoch)
            writer.add_scalar("LR", current_lr, epoch)
            writer.add_scalar("Time/epoch_sec", epoch_time, epoch)

            # Keep the best weights — cosine-to-zero means the last epoch is
            # usually the best, but this run is insurance against surprises.
            if test_acc > best_acc:
                best_acc = test_acc
                torch.save({
                    "epoch": epoch,
                    "test_acc": test_acc,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                }, f"checkpoints/{run_name}_best.pt")

            print(
                f"Epoch [{epoch + 1}/{epochs}], "
                f"Loss: {avg_epoch_loss:.4f}, "
                f"Train Acc: {train_acc:.4f}, "
                f"Test Acc: {test_acc:.4f}, "
                f"LR: {current_lr:.6f}, "
                f"{epoch_time:.1f}s"
            )
    finally:
        torch.save(model.state_dict(), f"checkpoints/{run_name}_final.pt")
        writer.close()

    print(f"Best test acc: {best_acc:.4f} (checkpoints/{run_name}_best.pt)")

#### Run - ViT

In [147]:
import json
from torch.utils.tensorboard import SummaryWriter

# One directory per run so TensorBoard curves never overlap between runs.
run_name = f"{datetime.now():%Y%m%d-%H%M%S}_ViT_simple"
writer = SummaryWriter(f"runs/cifar10/{run_name}")
writer.add_text(
    "config",
    json.dumps(
        {
            # Model
            "architecture": "ViT-Simple-CIFAR10",
            "image_size": IMAGE_SIZE,
            "channels": CHANNELS,
            "patch_size": PATCH_SIZE,
            "num_patches": (IMAGE_SIZE // PATCH_SIZE) ** 2,
            "num_classes": NUM_CLASSES,
            "embed_dim": EMBED_DIM,
            "num_heads": NUM_HEADS,
            "head_dim": EMBED_DIM // NUM_HEADS,
            "depth": DEPTH,
            "mlp_dim": MLP_DIM,
            "drop_rate": DROP_RATE,

            # Training
            "optimizer": "AdamW",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "scheduler": "warmup5+cosine",
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "label_smoothing": 0.1,
            "augmentation": "crop-reflect+hflip+erasing",
            "seed": SEED,
        },
        indent=2,
    ),
)

In [148]:
run(vit, optimizer=optimizer, scheduler=scheduler, criterion=criterion,
    epochs=EPOCHS, writer=writer, run_name=run_name)

Epoch [1/200], Loss: 1.9249, Train Acc: 0.4022, Test Acc: 0.4007, LR: 0.000084, 4.2s
Epoch [2/200], Loss: 1.7487, Train Acc: 0.4850, Test Acc: 0.4700, LR: 0.000138, 3.7s
Epoch [3/200], Loss: 1.6203, Train Acc: 0.5276, Test Acc: 0.5080, LR: 0.000192, 3.6s
Epoch [4/200], Loss: 1.5219, Train Acc: 0.5908, Test Acc: 0.5667, LR: 0.000246, 3.7s
Epoch [5/200], Loss: 1.4492, Train Acc: 0.6288, Test Acc: 0.6076, LR: 0.000300, 3.6s
Epoch [6/200], Loss: 1.4029, Train Acc: 0.6429, Test Acc: 0.6188, LR: 0.000300, 3.5s
Epoch [7/200], Loss: 1.3500, Train Acc: 0.6508, Test Acc: 0.6165, LR: 0.000300, 3.7s
Epoch [8/200], Loss: 1.3060, Train Acc: 0.6825, Test Acc: 0.6548, LR: 0.000300, 3.8s
Epoch [9/200], Loss: 1.2682, Train Acc: 0.6845, Test Acc: 0.6600, LR: 0.000300, 3.6s
Epoch [10/200], Loss: 1.2404, Train Acc: 0.6955, Test Acc: 0.6735, LR: 0.000300, 3.8s
Epoch [11/200], Loss: 1.2077, Train Acc: 0.7223, Test Acc: 0.6931, LR: 0.000299, 3.6s
Epoch [12/200], Loss: 1.1843, Train Acc: 0.7414, Test Acc: 0.70

#### Run - resnet

In [149]:
# import json
# from torch.utils.tensorboard import SummaryWriter

# # One directory per run so TensorBoard curves never overlap between runs.
# run_name = f"{datetime.now():%Y%m%d-%H%M%S}_resnet34"
# writer = SummaryWriter(f"runs/cifar10/{run_name}")
# writer.add_text("config", json.dumps({
#     "arch": "resnet34-cifar", "layers": [3, 4, 6, 3],
#     "optimizer": "AdamW", "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, 
#     "scheduler": "warmup5+cosine", "epochs": epochs, "batch_size": BATCH_SIZE,
#     "label_smoothing": 0.1, "augmentation": "crop-reflect+hflip+erasing", "seed": SEED,
# }))

# run(net, optimizer=optimizer, scheduler=scheduler, criterion=criterion,
#     epochs=epochs, writer=writer, run_name=run_name)

#### Run - CNN

In [150]:
# run(model)

## Plot Accuracy

In [151]:
# plt.plot(train_accuracies, label='Train Accuracy')
# plt.plot(test_accuracies, label='Test Accuracy')
# plt.xlabel('Epoch')
# plt.ylabel('Accuracy')
# plt.legend()
# plt.title('Training and Test Accuracy')
# plt.show()

## Benchmark

### Goal

CIFAR_10
* SOTA training acc = 94-96%
* Good test acc: 85%+
* Gap = around 5%–10% - Not use augmentation. 
* Gap = 2%–5% - With augmentation, BatchNorm, dropout, and weight decay.

### Fine-tunning Result

#### v1.0
Architecture + hyperparameters:

    * Flow = input -> conv(16,3,1,1) -> gelu -> maxpool(2,2) -> conv(32,3,1,1) -> gelu -> dropout(0.2) -> maxpool(2,2) -> conv(64,3,1,1) -> gelu -> dropout(0.3) -> maxpool(2,2) -> conv(64,3,1,1) -> gelu -> Flatten shape=(64, 64 * 4 * 4) -> Linear(64 * 4 * 4, 10) -> (CrossEntropyLoss)

    * Optimiser = AdamW
    
    * Learning rate = lr = LEARNING_RATE

    * # epoch = 20, batch_size = 64

    * Transform: normalising to [-1, 1]

    * Noise: 0.03

Result:
    
    Train Acc: 0.8922, Test Acc: 0.7565

Resnet34

Architecture + hyperparameters:

    * Flow (paper resnet-34 based) = input -> conv(64,7,2,3) -> batchnorm(64) -> relu -> maxpool(3,2,1) -> {residual block 3 x [conv(64,3,1,1) -> batchnorm(64) -> relu -> conv(64,3,1,1) -> batchnorm(64) -> relu] -> residual block 4 x [conv(128,3,2,1) -> batchnorm(128) -> relu -> conv(128,3,1,1) -> batchnorm(128) -> relu] -> residual block 6 x [conv(256,3,2,1) -> batchnorm(256) -> relu -> conv(256,3,1,1) -> batchnorm(256) -> relu] -> residual block 3 x [conv(512,3,2,1) -> batchnorm(512) -> relu -> conv(512,3,1,1) -> batchnorm(512) -> relu]} -> AdaptiveAvgPool2d(1,1) -> Flatten -> [Linear(512, 256) -> gelu -> Linear(256, 128) -> gelu -> Linear(128, 64) -> gelu -> Linear(64, 10)] -> (CrossEntropyLoss)

    * Optimiser = AdamW
    
    * Learning rate = lr = LEARNING_RATE, weight_decay=WEIGHT_DECAY

    * scheduler => 1e - 6

    * # epoch = 100, batch_size = 64

    * Transform: normalising to [-1, 1]

    * Noise: 0.03

Result:
    
    Train Acc: , Test Acc: 

#### v2.0 — canonical CIFAR ResNet-18 recipe (2026-07-17)

Changes vs the resnet34 v1 run (final: Train Acc 0.9251, Test Acc 0.8247, gap ~10%):

    * Stem: conv(64,3,1,1) + BN + ReLU, NO maxpool — the ImageNet stem (7x7 s2 + maxpool) was
      shrinking 32x32 inputs to 8x8 before the first block; stages now see 32-16-8-4 maps
    * Block: relu2 -> plain ReLU; post-add activation relu(identity + residual); bias-free convs;
      BN (not GroupNorm) in the downsample shortcut
    * Head: single Linear(512, 10) instead of the 512-256-128-64-10 GELU MLP
    * Depth: resnet18 [2,2,2,2] instead of resnet34 [3,4,6,3]
    * Init: Kaiming fan-out + zero-init of each block's last BN gamma (blocks start as identity)
    * Data: RandomCrop(reflect) + HFlip -> ToTensor -> Normalize(CIFAR mean/std) -> RandomErasing(p=0.5);
      removed VerticalFlip, the 2x-1 Lambda, and the in-loop Gaussian input noise
    * Optimiser: SGD lr=0.1, momentum=0.9, nesterov, wd=5e-4 on weights only (none on BN/biases);
      CrossEntropyLoss(label_smoothing=0.1); no gradient clipping
    * Schedule: 5-epoch linear warmup -> cosine to 0; # epoch = 100, batch_size = 128
    * Systems: AMP + channels_last + cudnn.benchmark; seeded (42); per-run TensorBoard dir;
      best/final checkpoints saved to checkpoints/
    * Eval: train accuracy on a clean fixed 10k subset (was: full augmented train set)

Result (verified run, 2026-07-17, RTX 5080, ~3.5 s/epoch, ~6 min total):

    Train Acc: 0.9997, Test Acc: 0.9561 (best), gap 4.4%
    (loss floors near ~0.52 because of label smoothing 0.1 — expected)

#### v3.0 — ViT from scratch, no pretraining (2026-08-12)

Architecture + hyperparameters (3.20M params):

    * Flow = input(3,32,32) -> PatchEmbedding[Conv2d(3,256,k=4,s=4) -> Flatten -> transpose
      -> concat CLS token -> + pos_embed] = (B, 65, 256)
      -> 6 x TransformerEncoderLayer[pre-LN: x + MSA(LN(x)) ; x + MLP(LN(x))]
         MSA = nn.MultiheadAttention(256, 8 heads, head_dim 32, dropout 0.1)
         MLP = Linear(256,512) -> GELU -> Drop(0.1) -> Linear(512,256) -> Drop(0.1)
      -> LayerNorm(256) -> take CLS token -> Linear(256, 10) -> (CrossEntropyLoss)
    * Init: trunc_normal_(std=0.02) on cls_token and pos_embed (learned, not sinusoidal)
    * Optimiser: AdamW, lr=3e-4, weight_decay=5e-2 (applied to ALL params — the decay/no_decay
      split is built in the cell but not passed to the optimizer)
    * Schedule: 5-epoch linear warmup (start_factor 0.1) -> cosine to 0; # epoch = 200, batch_size = 128
    * Loss: CrossEntropyLoss(label_smoothing=0.1) -> loss floors near ~0.50, expected
    * Data: RandomCrop(32, pad 4, reflect) + HFlip -> ToTensor -> Normalize(CIFAR mean/std).
      RandomErasing is COMMENTED OUT for this run (the logged config string still says
      "crop-reflect+hflip+erasing" — stale label, fix before comparing to the ResNet runs)
    * Systems: AMP + channels_last + cudnn.benchmark; seeded (42); per-run TensorBoard dir
    * Eval: train accuracy on the clean fixed 10k train subset

Result (verified run `20260812-174242_ViT_simple`, RTX 5080, ~3.6 s/epoch, ~12 min total):

    Best test acc: 0.8435 (epoch 194) — Train Acc 1.0000, gap ~15.6%
    Final (epoch 200): Loss 0.5057, Train Acc 1.0000, Test Acc 0.8426

Trajectory:

    | epoch | loss   | train  | test   |
    |-------|--------|--------|--------|
    | 50    | 0.7349 | 0.9442 | 0.8160 |
    | 87    | 0.5899 | 0.9909 | 0.8272 |  <- train saturates here
    | 100   | 0.5667 | 0.9918 | 0.8214 |
    | 125   | 0.5345 | 0.9979 | 0.8317 |
    | 150   | 0.5167 | 0.9997 | 0.8362 |
    | 200   | 0.5057 | 1.0000 | 0.8426 |

Read:

    * Underperforms the v2.0 ResNet-18 (95.6% test) by ~11 points, and misses the 85% goal.
      Expected: ViTs have no convolutional locality/translation prior, so on 50k images they
      lean on augmentation and regularization that this run mostly turned off.
    * Train hits 100% by ~epoch 87 while test stalls in the low 82s — the model memorizes the
      train split and the remaining 113 epochs buy only ~1.5 points (cosine decay polish).
      The 15.6% gap is pure overfitting, far outside the 2-5% augmented-regime target.

Next levers (in rough order of expected payoff):

    * Turn RandomErasing back on, and add Mixup/CutMix + RandAugment — the standard
      "ViT on small data" recipe; this is the biggest single gap.
    * Stochastic depth (drop_path ~0.1) across the 6 blocks; raise drop_rate above 0.1.
    * Exclude LayerNorm gains, biases, cls_token and pos_embed from weight decay
      (the decay/no_decay lists are already built — just pass them as param groups).
    * Shrink to depth 4-6 / embed 192 or try patch_size=2 (256 tokens) for finer detail.


In [19]:
data = [1,3,3,3,2,2,2,4]
temp = sorted(data)
max(temp, key=data.count)

2